# 🚗 Vehicle 3D Reconstruction Pipeline - Google Drive Batch Processing

This pipeline reads an organized vehicle dataset from Google Drive and, for each make/model/year:
1. **Preprocessing**: Background removal and grayscale conversion
2. **Orientation Classification**: Determine vehicle orientation (8 directions)
3. **3D Reconstruction**: Build a point cloud with HLOC + COLMAP
4. **Export**: Save results to Drive as `{make}_{model}_{year}.zip`

---

**Dataset Structure (Input):**
```
organized/
├── sedan/
│   ├── Audi/
│   │   ├── A3/
│   │   │   └── 2013/*.jpg
│   │   └── A6L/
│   │       └── 2014/*.jpg
├── SUV/
│   └── ...
```

**Output Structure:**
```
processed/
├── sedan/
│   ├── Audi/
│   │   ├── A3/
│   │   │   └── Audi_A3_2013.zip
│   │   └── A6L/
│   │       └── Audi_A6L_2014.zip
```

---
## 📦 Step 0: Setup and Google Drive Connection

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted")

In [ ]:
# ============================================
# 🔧 SETTINGS - EDIT THIS CELL
# ============================================

# Organized dataset folder on Google Drive
DRIVE_INPUT_PATH = "/content/drive/MyDrive/organized-dataset"

# Folder where results will be saved
DRIVE_OUTPUT_PATH = "/content/drive/MyDrive/processed-organized-dataset"

# Model file (for the orientation classifier)
MODEL_PATH = "/content/model_best.pth"

# Classification confidence threshold
CONFIDENCE_THRESHOLD = 0.75

# Process a specific car type (None = process all)
# Example: "sedan", "SUV", "hatchback", None
FILTER_CAR_TYPE = "sports"

# Process a specific make (None = process all)
# Example: "Audi", "BMW", None
FILTER_MAKE = None

# Process a specific model (None = process all)
# Example: "A3", "3 Series", None
FILTER_MODEL = None

print("✅ Settings loaded")
print(f"   📁 Input:  {DRIVE_INPUT_PATH}")
print(f"   📁 Output: {DRIVE_OUTPUT_PATH}")
print(f"   🧠 Model:  {MODEL_PATH}")

In [ ]:
# HLOC installation
import os
os.chdir('/content')

# Clone and install HLOC
if not os.path.exists('/content/Hierarchical-Localization'):
    !git clone --quiet --recursive https://github.com/cvg/Hierarchical-Localization/
    %cd Hierarchical-Localization/
    !pip install -q -e .
else:
    print("HLOC already installed")

# COLMAP installation
!apt-get install -qq colmap

%cd /content
print("✅ HLOC and COLMAP installation complete")

In [ ]:
# Install core libraries - ROBUST INSTALL
print("📦 Installing libraries...\n")

# 1. Fix Numpy
print("1️⃣ Installing Numpy...")
!pip uninstall -y numpy -q
!pip install numpy==1.26.4 -q

# 2. Rembg and dependencies
print("2️⃣ Installing Rembg...")
!pip install rembg==2.0.55 -q
!pip install onnxruntime -q

# 3. Other core libraries
print("3️⃣ Installing OpenCV and PIL...")
!pip install pillow opencv-python-headless tqdm -q

# 4. PyTorch
print("4️⃣ Installing PyTorch...")
!pip install torch torchvision scikit-learn -q

print("\n✅ All libraries installed!")
print("⚠️  Please do Runtime > Restart runtime, then run the next cell\n")

---
## 🖼️ Preprocessing Functions

In [ ]:
import os
import io
from glob import glob
from pathlib import Path
from tqdm import tqdm
import numpy as np
import cv2
from PIL import Image, ImageOps
from rembg import remove

# ==== Preprocessing Settings ====
CROP_MARGIN_RATIO = 0.03
MASK_ERODE_PX = 1
MASK_BLUR_SIGMA = 1.0
HISTEQ_MODE = "clahe"
CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID_SIZE = (8, 8)
SET_BACKGROUND_BLACK = False

def load_image_fix_orientation(path):
    """Load image and fix EXIF orientation"""
    im = Image.open(path).convert("RGBA")
    im = ImageOps.exif_transpose(im)
    return im

def pil_to_bytes_rgba(img: Image.Image) -> bytes:
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()

def remove_background(img_rgba: Image.Image) -> Image.Image:
    data = pil_to_bytes_rgba(img_rgba)
    out = remove(data)
    return Image.open(io.BytesIO(out)).convert("RGBA")

def mask_from_rgba(rgba_img: Image.Image) -> np.ndarray:
    a = np.array(rgba_img.split()[-1])
    return a

def largest_contour_bbox(mask: np.ndarray):
    binm = (mask > 0).astype(np.uint8) * 255
    cnts, _ = cv2.findContours(binm, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    return cv2.boundingRect(c)

def pad_bbox(x, y, w, h, img_w, img_h, pad_px):
    x2 = max(0, x - pad_px)
    y2 = max(0, y - pad_px)
    x3 = min(img_w, x + w + pad_px)
    y3 = min(img_h, y + h + pad_px)
    return x2, y2, x3 - x2, y3 - y2

def apply_histeq_gray(gray: np.ndarray):
    if HISTEQ_MODE == "plain":
        return cv2.equalizeHist(gray)
    elif HISTEQ_MODE == "clahe":
        clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT, tileGridSize=CLAHE_TILE_GRID_SIZE)
        return clahe.apply(gray)
    return gray

def soften_mask(mask: np.ndarray, erode_px=1, blur_sigma=1.0):
    m = mask.copy()
    if erode_px > 0:
        kernel = np.ones((3,3), np.uint8)
        m = cv2.erode(m, kernel, iterations=erode_px)
    if blur_sigma and blur_sigma > 0:
        k = int(blur_sigma*3)*2 + 1
        m = cv2.GaussianBlur(m, (k,k), blur_sigma)
    return m

def preprocess_vehicle_images(input_dir, output_dir, silent=False):
    """
    Process vehicle images:
    1. Fix EXIF orientation
    2. Background removal (rembg)
    3. Crop + padding
    4. Grayscale + CLAHE
    5. Mask softening
    """
    input_path = Path(input_dir)
    final_dir = Path(output_dir) / 'final'
    final_dir.mkdir(parents=True, exist_ok=True)

    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif', '.webp'}
    image_files = [f for f in input_path.iterdir() if f.is_file() and f.suffix.lower() in valid_extensions]

    if not image_files:
        return 0, 0

    if not silent:
        print(f"   🖼️  Preprocessing: {len(image_files)} images")

    success_count = 0
    error_count = 0

    for img_file in tqdm(image_files, desc="   Preprocessing", disable=silent):
        try:
            name = img_file.stem
            pil_rgba = load_image_fix_orientation(img_file)
            pil_nobg = remove_background(pil_rgba)
            mask = mask_from_rgba(pil_nobg)

            H, W = mask.shape
            bbox = largest_contour_bbox(mask)
            if bbox is None:
                error_count += 1
                continue

            x, y, w, h = bbox
            pad_px = int(min(W, H) * CROP_MARGIN_RATIO)
            x, y, w, h = pad_bbox(x, y, w, h, W, H, pad_px)

            pil_nobg_np = np.array(pil_nobg)
            crop_rgba = pil_nobg_np[y:y+h, x:x+w, :]
            crop_mask = mask[y:y+h, x:x+w]

            crop_bgra = cv2.cvtColor(crop_rgba, cv2.COLOR_RGBA2BGRA)
            crop_bgr = cv2.cvtColor(crop_bgra, cv2.COLOR_BGRA2BGR)
            gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)

            soft_m = soften_mask(crop_mask, erode_px=MASK_ERODE_PX, blur_sigma=MASK_BLUR_SIGMA).astype(np.float32) / 255.0
            soft_m_3 = np.stack([soft_m]*3, axis=-1)

            gray_eq = apply_histeq_gray(gray)
            eq_rgb = cv2.cvtColor(gray_eq, cv2.COLOR_GRAY2BGR)

            if SET_BACKGROUND_BLACK:
                bg_rgb = np.zeros_like(eq_rgb)
            else:
                bg_rgb = crop_bgr.copy()

            comp_rgb = (eq_rgb * soft_m_3 + bg_rgb * (1.0 - soft_m_3)).astype(np.uint8)
            final_bgra = cv2.cvtColor(comp_rgb, cv2.COLOR_BGR2BGRA)
            final_bgra[:,:,3] = crop_mask

            final_path = final_dir / f"{name}_final.png"
            cv2.imwrite(str(final_path), final_bgra)
            success_count += 1

        except Exception as e:
            error_count += 1
            continue

    return success_count, error_count

print("✅ Preprocessing functions loaded")

---
## 🧭 Classification Functions

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import json
import shutil
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

class VehicleOrientationClassifier:
    ORIENTATIONS = [
        'front', 'front-left', 'left', 'rear-left',
        'rear', 'rear-right', 'right', 'front-right'
    ]

    def __init__(self, model_path):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.orientation_to_idx = {o: i for i, o in enumerate(self.ORIENTATIONS)}
        self.idx_to_orientation = {i: o for o, i in self.orientation_to_idx.items()}

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.Lambda(lambda x: x.convert('RGB') if x.mode != 'RGB' else x),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

        self.model = self._create_model()

        if not os.path.exists(model_path):
            raise FileNotFoundError(f"❌ Model file not found: {model_path}")

        self.model.load_state_dict(torch.load(model_path, map_location=self.device))
        self.model.eval()

    def _create_model(self):
        from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
        weights = EfficientNet_B0_Weights.DEFAULT
        model = efficientnet_b0(weights=weights)
        num_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(num_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(256, len(self.ORIENTATIONS))
        )
        return model.to(self.device)

    def classify_single(self, image_path):
        image = Image.open(image_path)
        image_tensor = self.transform(image).unsqueeze(0).to(self.device)
        with torch.no_grad():
            outputs = self.model(image_tensor)
            probabilities = torch.softmax(outputs, dim=1)
            confidence, predicted = torch.max(probabilities, 1)
        return self.idx_to_orientation[predicted.item()], confidence.item()

    def classify_batch(self, input_dir, output_dir, confidence_threshold=0.75, silent=False):
        input_path = Path(input_dir)
        output_path = Path(output_dir)

        for orientation in self.ORIENTATIONS:
            (output_path / orientation).mkdir(parents=True, exist_ok=True)
        (output_path / 'low_confidence').mkdir(parents=True, exist_ok=True)

        valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}
        image_files = [f for f in input_path.iterdir() if f.is_file() and f.suffix.lower() in valid_extensions]

        if not image_files:
            return {}

        if not silent:
            print(f"   🧭 Classification: {len(image_files)} images")

        results = {orientation: [] for orientation in self.ORIENTATIONS}
        results['low_confidence'] = []

        for img_file in tqdm(image_files, desc="   Classification", disable=silent):
            try:
                orientation, confidence = self.classify_single(img_file)

                if confidence < confidence_threshold:
                    target_dir = output_path / 'low_confidence'
                    category = 'low_confidence'
                else:
                    target_dir = output_path / orientation
                    category = orientation

                shutil.copy2(img_file, target_dir / img_file.name)
                results[category].append({'filename': img_file.name, 'confidence': round(confidence, 3)})

            except Exception as e:
                continue

        return results

print("✅ Classification functions loaded")

---
## 🏗️ Reconstruction Functions

In [ ]:
import sys
sys.path.insert(0, '/content/Hierarchical-Localization')

from pathlib import Path
from collections import defaultdict
import shutil

try:
    from hloc import extract_features, match_features, reconstruction
    from hloc.utils import viz_3d
    import pycolmap
    HLOC_AVAILABLE = True
except ImportError:
    HLOC_AVAILABLE = False
    print("⚠️ HLOC not loaded yet; run the HLOC installation for reconstruction")

ORIENTATIONS_ORDER = [
    "front", "front-left", "left", "rear-left",
    "rear", "rear-right", "right", "front-right"
]
NEIGHBOR_RING = 1

# ==== GLOBAL MODEL CACHE ====
_FEATURE_CONF = None
_MATCHER_CONF = None
_MODELS_LOADED = False

def init_hloc_models():
    """Load HLOC models once and cache them"""
    global _FEATURE_CONF, _MATCHER_CONF, _MODELS_LOADED

    if _MODELS_LOADED:
        return _FEATURE_CONF, _MATCHER_CONF

    print("🔄 Loading HLOC models (once)...")

    _FEATURE_CONF = extract_features.confs['superpoint_aachen'].copy()
    _FEATURE_CONF['preprocessing']['resize_max'] = 1024

    _MATCHER_CONF = match_features.confs['superglue'].copy()

    # Preload the SuperGlue model (warmup)
    import torch
    from hloc.matchers.superglue import SuperGlue
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    _ = SuperGlue(_MATCHER_CONF['model']).eval().to(device)

    _MODELS_LOADED = True
    print("✅ HLOC models loaded and cached!")

    return _FEATURE_CONF, _MATCHER_CONF

def flatten_images(classified_dir, flat_dir):
    flat_path = Path(flat_dir)
    flat_path.mkdir(parents=True, exist_ok=True)
    classified_path = Path(classified_dir)
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

    total_copied = 0
    angle_counts = {}

    for orientation in ORIENTATIONS_ORDER:
        orientation_dir = classified_path / orientation
        if not orientation_dir.exists():
            continue

        images = [f for f in orientation_dir.iterdir() if f.is_file() and f.suffix.lower() in valid_extensions]
        count = 0
        for img in images:
            new_name = f"{orientation}__{img.name}"
            shutil.copy2(img, flat_path / new_name)
            count += 1
            total_copied += 1
        angle_counts[orientation] = count

    return total_copied, angle_counts

def get_angle_from_filename(filename):
    if "__" in filename:
        return filename.split("__", 1)[0].lower()
    return None

def get_neighbor_angles(angle, ring=1):
    if angle not in ORIENTATIONS_ORDER:
        return []
    i = ORIENTATIONS_ORDER.index(angle)
    N = len(ORIENTATIONS_ORDER)
    neighbors = []
    for d in range(1, ring + 1):
        neighbors.append(ORIENTATIONS_ORDER[(i + d) % N])
        neighbors.append(ORIENTATIONS_ORDER[(i - d) % N])
    return neighbors

def create_pairs_with_neighbors(flat_dir, output_file):
    flat_path = Path(flat_dir)
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

    by_angle = defaultdict(list)
    for img_file in sorted(flat_path.iterdir()):
        if img_file.is_file() and img_file.suffix.lower() in valid_extensions:
            angle = get_angle_from_filename(img_file.name)
            if angle:
                by_angle[angle].append(img_file.name)

    pairs = set()

    for angle, img_list in by_angle.items():
        for i in range(len(img_list)):
            for j in range(i + 1, len(img_list)):
                pairs.add((img_list[i], img_list[j]))

    for angle, img_list in by_angle.items():
        neighbors = get_neighbor_angles(angle, ring=NEIGHBOR_RING)
        for neighbor_angle in neighbors:
            if neighbor_angle in by_angle:
                for img1 in img_list:
                    for img2 in by_angle[neighbor_angle]:
                        if img1 != img2:
                            pair = tuple(sorted([img1, img2]))
                            pairs.add(pair)

    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, 'w') as f:
        for img1, img2 in sorted(pairs):
            f.write(f"{img1} {img2}\n")

    return len(pairs)

def run_full_reconstruction(classified_dir, output_dir, silent=False):
    if not HLOC_AVAILABLE:
        raise RuntimeError("HLOC is not installed!")

    # Get cached models
    feature_conf, matcher_conf = init_hloc_models()

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    flat_dir = output_path / 'images_flat'
    pairs_file = output_path / 'pairs.txt'
    features_file = output_path / 'features.h5'
    matches_file = output_path / 'matches.h5'
    sfm_dir = output_path / 'sfm'

    try:
        total_images, angle_counts = flatten_images(classified_dir, flat_dir)
        if total_images < 2:
            return None

        num_pairs = create_pairs_with_neighbors(flat_dir, pairs_file)
        if num_pairs == 0:
            return None

        if not silent:
            print(f"   🔍 Feature extraction...")

        extract_features.main(
            feature_conf,
            flat_dir,
            image_list=None,
            feature_path=features_file,
            as_half=True
        )

        if not silent:
            print(f"   🎯 Feature matching...")

        match_features.main(
            matcher_conf,
            pairs_file,
            features=features_file,
            matches=matches_file
        )

        if not silent:
            print(f"   🏗️  COLMAP reconstruction...")

        model = reconstruction.main(
            sfm_dir,
            flat_dir,
            pairs_file,
            features_file,
            matches_file,
            image_list=None
        )

        return {
            'model': model,
            'total_images': total_images,
            'registered_images': len(model.images) if model else 0,
            'points_3d': len(model.points3D) if model else 0,
            'num_pairs': num_pairs,
            'angle_counts': angle_counts,
            'sfm_dir': sfm_dir
        }

    except Exception as e:
        print(f"   ❌ Reconstruction error: {str(e)}")
        return None

print("✅ Reconstruction functions loaded (model caching enabled)")

---
## 🚀 Batch Processing Pipeline

In [ ]:
import sys
sys.path.insert(0, '/content/Hierarchical-Localization')

from pathlib import Path
from collections import defaultdict
from functools import partial
from queue import Queue
from threading import Thread
import shutil

import h5py
import torch
from tqdm import tqdm

try:
    from hloc import extract_features, reconstruction, matchers, logger
    from hloc.utils.base_model import dynamic_load
    from hloc.utils.parsers import names_to_pair, parse_retrieval
    from hloc.match_features import FeaturePairsDataset, writer_fn, find_unique_new_pairs, WorkQueue
    import pycolmap
    HLOC_AVAILABLE = True
except ImportError:
    HLOC_AVAILABLE = False
    print("⚠️ HLOC not loaded yet")

ORIENTATIONS_ORDER = [
    "front", "front-left", "left", "rear-left",
    "rear", "rear-right", "right", "front-right"
]
NEIGHBOR_RING = 1

# ==== GLOBAL MODEL CACHE ====
_FEATURE_CONF = None
_MATCHER_MODEL = None
_DEVICE = None
_MODELS_LOADED = False

def init_hloc_models():
    """Load HLOC models once and cache them"""
    global _FEATURE_CONF, _MATCHER_MODEL, _DEVICE, _MODELS_LOADED

    if _MODELS_LOADED:
        return

    print("🔄 Loading HLOC models (once)...")

    _DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # Feature config
    _FEATURE_CONF = extract_features.confs['superpoint_aachen'].copy()
    _FEATURE_CONF['preprocessing']['resize_max'] = 1024

    # Matcher model - LOAD ONCE
    matcher_conf = {"name": "superglue", "weights": "outdoor", "sinkhorn_iterations": 50}
    Model = dynamic_load(matchers, matcher_conf["name"])
    _MATCHER_MODEL = Model(matcher_conf).eval().to(_DEVICE)

    _MODELS_LOADED = True
    print(f"✅ HLOC models loaded! (device: {_DEVICE})")

@torch.no_grad()
def match_features_cached(pairs_path, match_path, feature_path_q, feature_path_ref=None):
    """Feature matching with the cached model"""
    global _MATCHER_MODEL, _DEVICE

    if feature_path_ref is None:
        feature_path_ref = feature_path_q

    match_path.parent.mkdir(exist_ok=True, parents=True)

    pairs = parse_retrieval(pairs_path)
    pairs = [(q, r) for q, rs in pairs.items() for r in rs]
    pairs = find_unique_new_pairs(pairs, match_path)

    if len(pairs) == 0:
        return

    dataset = FeaturePairsDataset(pairs, feature_path_q, feature_path_ref)
    loader = torch.utils.data.DataLoader(
        dataset, num_workers=5, batch_size=1, shuffle=False, pin_memory=True
    )
    writer_queue = WorkQueue(partial(writer_fn, match_path=match_path), 5)

    for idx, data in enumerate(tqdm(loader, smoothing=0.1, desc="   Matching")):
        data = {
            k: v if k.startswith("image") else v.to(_DEVICE, non_blocking=True)
            for k, v in data.items()
        }
        pred = _MATCHER_MODEL(data)
        pair = names_to_pair(*pairs[idx])
        writer_queue.put((pair, pred))
    writer_queue.join()

def flatten_images(classified_dir, flat_dir):
    flat_path = Path(flat_dir)
    flat_path.mkdir(parents=True, exist_ok=True)
    classified_path = Path(classified_dir)
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

    total_copied = 0
    angle_counts = {}

    for orientation in ORIENTATIONS_ORDER:
        orientation_dir = classified_path / orientation
        if not orientation_dir.exists():
            continue

        images = [f for f in orientation_dir.iterdir() if f.is_file() and f.suffix.lower() in valid_extensions]
        count = 0
        for img in images:
            new_name = f"{orientation}__{img.name}"
            shutil.copy2(img, flat_path / new_name)
            count += 1
            total_copied += 1
        angle_counts[orientation] = count

    return total_copied, angle_counts

def get_angle_from_filename(filename):
    if "__" in filename:
        return filename.split("__", 1)[0].lower()
    return None

def get_neighbor_angles(angle, ring=1):
    if angle not in ORIENTATIONS_ORDER:
        return []
    i = ORIENTATIONS_ORDER.index(angle)
    N = len(ORIENTATIONS_ORDER)
    neighbors = []
    for d in range(1, ring + 1):
        neighbors.append(ORIENTATIONS_ORDER[(i + d) % N])
        neighbors.append(ORIENTATIONS_ORDER[(i - d) % N])
    return neighbors

def create_pairs_with_neighbors(flat_dir, output_file):
    flat_path = Path(flat_dir)
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

    by_angle = defaultdict(list)
    for img_file in sorted(flat_path.iterdir()):
        if img_file.is_file() and img_file.suffix.lower() in valid_extensions:
            angle = get_angle_from_filename(img_file.name)
            if angle:
                by_angle[angle].append(img_file.name)

    pairs = set()

    for angle, img_list in by_angle.items():
        for i in range(len(img_list)):
            for j in range(i + 1, len(img_list)):
                pairs.add((img_list[i], img_list[j]))

    for angle, img_list in by_angle.items():
        neighbors = get_neighbor_angles(angle, ring=NEIGHBOR_RING)
        for neighbor_angle in neighbors:
            if neighbor_angle in by_angle:
                for img1 in img_list:
                    for img2 in by_angle[neighbor_angle]:
                        if img1 != img2:
                            pair = tuple(sorted([img1, img2]))
                            pairs.add(pair)

    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, 'w') as f:
        for img1, img2 in sorted(pairs):
            f.write(f"{img1} {img2}\n")

    return len(pairs)

def run_full_reconstruction(classified_dir, output_dir, silent=False):
    if not HLOC_AVAILABLE:
        raise RuntimeError("HLOC is not installed!")

    # Load models (on first call)
    init_hloc_models()

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    flat_dir = output_path / 'images_flat'
    pairs_file = output_path / 'pairs.txt'
    features_file = output_path / 'features.h5'
    matches_file = output_path / 'matches.h5'
    sfm_dir = output_path / 'sfm'

    try:
        total_images, angle_counts = flatten_images(classified_dir, flat_dir)
        if total_images < 2:
            return None

        num_pairs = create_pairs_with_neighbors(flat_dir, pairs_file)
        if num_pairs == 0:
            return None

        if not silent:
            print(f"   🔍 Feature extraction...")

        extract_features.main(
            _FEATURE_CONF,
            flat_dir,
            image_list=None,
            feature_path=features_file,
            as_half=True
        )

        if not silent:
            print(f"   🎯 Feature matching (cached model)...")

        # USE CACHED MODEL
        match_features_cached(
            pairs_path=pairs_file,
            match_path=matches_file,
            feature_path_q=features_file
        )

        if not silent:
            print(f"   🏗️  COLMAP reconstruction...")

        model = reconstruction.main(
            sfm_dir,
            flat_dir,
            pairs_file,
            features_file,
            matches_file,
            image_list=None
        )

        return {
            'model': model,
            'total_images': total_images,
            'registered_images': len(model.images) if model else 0,
            'points_3d': len(model.points3D) if model else 0,
            'num_pairs': num_pairs,
            'angle_counts': angle_counts,
            'sfm_dir': sfm_dir
        }

    except Exception as e:
        print(f"   ❌ Reconstruction error: {str(e)}")
        return None

print("✅ Reconstruction functions loaded (REAL model caching enabled)")

In [ ]:
import time
import zipfile
from datetime import datetime

def clean_name_for_path(name):
    """Sanitize characters for a safe file name"""
    invalid_chars = '<>:"/\\|?* '
    for char in invalid_chars:
        name = name.replace(char, '_')
    return name.strip('_')

def discover_vehicles(input_path):
    """
    Discover all vehicles in the organized dataset
    Returns: list of (car_type, make, model, year, full_path)
    """
    vehicles = []
    input_path = Path(input_path)

    if not input_path.exists():
        raise FileNotFoundError(f"❌ Input folder not found: {input_path}")

    # Scan car_type folders
    for car_type_dir in sorted(input_path.iterdir()):
        if not car_type_dir.is_dir():
            continue
        car_type = car_type_dir.name

        # Scan make folders
        for make_dir in sorted(car_type_dir.iterdir()):
            if not make_dir.is_dir():
                continue
            make = make_dir.name

            # Scan model folders
            for model_dir in sorted(make_dir.iterdir()):
                if not model_dir.is_dir():
                    continue
                model = model_dir.name

                # Scan year folders
                for year_dir in sorted(model_dir.iterdir()):
                    if not year_dir.is_dir():
                        continue
                    year = year_dir.name

                    # Check whether images exist
                    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}
                    images = [f for f in year_dir.iterdir() if f.is_file() and f.suffix.lower() in valid_extensions]

                    if images:
                        vehicles.append({
                            'car_type': car_type,
                            'make': make,
                            'model': model,
                            'year': year,
                            'path': year_dir,
                            'image_count': len(images)
                        })

    return vehicles

def create_result_zip(source_dir, zip_path):
    """Compress reconstruction results into a zip file"""
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        source_path = Path(source_dir)
        for file in source_path.rglob('*'):
            if file.is_file():
                arcname = file.relative_to(source_path)
                zipf.write(file, arcname)
    return zip_path

def process_single_vehicle(vehicle, classifier, work_dir, output_base_path, confidence_threshold=0.75):
    """
    Process a single vehicle:
    1. Preprocessing
    2. Classification
    3. Reconstruction
    4. Create zip and save to Drive
    """
    car_type = vehicle['car_type']
    make = vehicle['make']
    model = vehicle['model']
    year = vehicle['year']
    input_path = vehicle['path']

    # Working directories
    vehicle_work_dir = Path(work_dir) / f"{make}_{model}_{year}"
    preprocessed_dir = vehicle_work_dir / 'preprocessed'
    classified_dir = vehicle_work_dir / 'classified'
    reconstruction_dir = vehicle_work_dir / 'reconstruction'

    # Output directory and zip name
    # Output: processed/{car_type}/{make}/{model}/{make}_{model}_{year}.zip
    output_dir = Path(output_base_path) / car_type / make / model
    output_dir.mkdir(parents=True, exist_ok=True)

    zip_name = f"{clean_name_for_path(make)}_{clean_name_for_path(model)}_{year}.zip"
    zip_path = output_dir / zip_name

    # Check whether already processed
    if zip_path.exists():
        return {'status': 'skipped', 'reason': 'already_exists', 'zip_path': str(zip_path)}

    try:
        # Start clean
        if vehicle_work_dir.exists():
            shutil.rmtree(vehicle_work_dir)
        vehicle_work_dir.mkdir(parents=True, exist_ok=True)

        # STEP 1: Preprocessing
        success_count, error_count = preprocess_vehicle_images(
            input_dir=input_path,
            output_dir=preprocessed_dir,
            silent=True
        )

        if success_count == 0:
            return {'status': 'failed', 'reason': 'preprocessing_failed'}

        # STEP 2: Classification
        classification_results = classifier.classify_batch(
            input_dir=f"{preprocessed_dir}/final",
            output_dir=classified_dir,
            confidence_threshold=confidence_threshold,
            silent=True
        )

        # STEP 3: Reconstruction
        reconstruction_results = run_full_reconstruction(
            classified_dir=classified_dir,
            output_dir=reconstruction_dir,
            silent=True
        )

        # STEP 4: Create zip and save
        create_result_zip(reconstruction_dir, zip_path)

        # Cleanup
        shutil.rmtree(vehicle_work_dir)

        return {
            'status': 'success',
            'zip_path': str(zip_path),
            'preprocessing': {'success': success_count, 'errors': error_count},
            'reconstruction': reconstruction_results
        }

    except Exception as e:
        # Cleanup on error
        if vehicle_work_dir.exists():
            shutil.rmtree(vehicle_work_dir)
        return {'status': 'failed', 'reason': str(e)}


def run_batch_pipeline(
    input_path,
    output_path,
    model_path,
    confidence_threshold=0.75,
    filter_car_type=None,
    filter_make=None,
    filter_model=None
):
    """
    Process all vehicles in batch
    """
    start_time = time.time()
    work_dir = Path('/content/pipeline_work')

    print("\n" + "="*70)
    print("🚗 VEHICLE 3D RECONSTRUCTION - BATCH PIPELINE")
    print("="*70)

    # Discover vehicles
    print("\n📂 Discovering vehicles...")
    vehicles = discover_vehicles(input_path)

    # Filtering
    if filter_car_type:
        vehicles = [v for v in vehicles if v['car_type'].lower() == filter_car_type.lower()]
    if filter_make:
        vehicles = [v for v in vehicles if v['make'].lower() == filter_make.lower()]
    if filter_model:
        vehicles = [v for v in vehicles if v['model'].lower() == filter_model.lower()]

    if not vehicles:
        print("❌ No vehicles matched the filters!")
        return

    print(f"✅ {len(vehicles)} vehicles found")

    # Show summary
    print("\n📋 Vehicles to process:")
    car_types = set(v['car_type'] for v in vehicles)
    makes = set(v['make'] for v in vehicles)
    print(f"   Car types: {len(car_types)}")
    print(f"   Makes: {len(makes)}")
    print(f"   Total model/year: {len(vehicles)}")

    # Load classifier (once)
    print(f"\n🧠 Loading model: {model_path}")
    classifier = VehicleOrientationClassifier(model_path=model_path)
    print("✅ Model loaded!")

    # Start processing
    print("\n" + "="*70)
    print("🚀 BATCH PROCESSING STARTING")
    print("="*70 + "\n")

    results = {
        'success': [],
        'failed': [],
        'skipped': []
    }

    for i, vehicle in enumerate(vehicles, 1):
        v_name = f"{vehicle['make']} {vehicle['model']} ({vehicle['year']})"
        print(f"\n[{i}/{len(vehicles)}] 🚗 {v_name}")
        print(f"   📁 {vehicle['car_type']} | {vehicle['image_count']} images")

        result = process_single_vehicle(
            vehicle=vehicle,
            classifier=classifier,
            work_dir=work_dir,
            output_base_path=output_path,
            confidence_threshold=confidence_threshold
        )

        if result['status'] == 'success':
            print(f"   ✅ Success: {result['zip_path']}")
            results['success'].append({**vehicle, **result})
        elif result['status'] == 'skipped':
            print(f"   ⏭️  Skipped (already exists)")
            results['skipped'].append({**vehicle, **result})
        else:
            print(f"   ❌ Error: {result.get('reason', 'unknown')}")
            results['failed'].append({**vehicle, **result})

    # Temizlik
    if work_dir.exists():
        shutil.rmtree(work_dir)

    # Summary
    elapsed_time = time.time() - start_time

    print("\n" + "="*70)
    print("🎉 BATCH PROCESSING COMPLETE!")
    print("="*70)
    print(f"\n⏱️  Total time: {elapsed_time:.1f} seconds ({elapsed_time/60:.1f} minutes)")
    print(f"\n📊 Results:")
    print(f"   ✅ Success: {len(results['success'])}")
    print(f"   ⏭️  Skipped: {len(results['skipped'])}")
    print(f"   ❌ Failed: {len(results['failed'])}")
    print(f"\n📁 Output: {output_path}")
    print("="*70 + "\n")

    return results

print("✅ Batch processing functions loaded")

---
## ▶️ RUN THE PIPELINE

In [ ]:
# Discover the dataset and show a summary
vehicles = discover_vehicles(DRIVE_INPUT_PATH)

print(f"\n📊 Dataset Summary:")
print(f"   Total vehicles: {len(vehicles)}")

# Car type distribution
from collections import Counter
type_counts = Counter(v['car_type'] for v in vehicles)
make_counts = Counter(v['make'] for v in vehicles)

print(f"\n📋 Car Type Distribution:")
for t, c in type_counts.most_common(10):
    print(f"   {t}: {c}")

print(f"\n🏭 Top 10 Makes:")
for m, c in make_counts.most_common(10):
    print(f"   {m}: {c}")

In [ ]:
# 🚀 RUN THE BATCH PIPELINE
results = run_batch_pipeline(
    input_path=DRIVE_INPUT_PATH,
    output_path=DRIVE_OUTPUT_PATH,
    model_path=MODEL_PATH,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    filter_car_type=FILTER_CAR_TYPE,
    filter_make=FILTER_MAKE,
    filter_model=FILTER_MODEL
)

---
## 📊 Review Results

In [ ]:
# Show successful results
if results and results['success']:
    print("\n✅ Successful:")
    for r in results['success'][:20]:  # First 20
        print(f"   {r['make']} {r['model']} ({r['year']}) → {r['zip_path']}")

if results and results['failed']:
    print("\n❌ Failed:")
    for r in results['failed'][:20]:
        print(f"   {r['make']} {r['model']} ({r['year']}) → {r.get('reason', 'unknown')}")

In [ ]:
# Show output folder structure
!echo "\n📁 OUTPUT FOLDER STRUCTURE:"
!find {DRIVE_OUTPUT_PATH} -name "*.zip" | head -30

---
## 🔧 Test a Single Vehicle

You can process a single vehicle to test the pipeline.

In [ ]:
# Test a single vehicle
TEST_CAR_TYPE = "sports"  # Change this
TEST_MAKE = "Toyota"       # Change this
TEST_MODEL = "86"        # Change this

results = run_batch_pipeline(
    input_path=DRIVE_INPUT_PATH,
    output_path=DRIVE_OUTPUT_PATH,
    model_path=MODEL_PATH,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    filter_car_type=TEST_CAR_TYPE,
    filter_make=TEST_MAKE,
    filter_model=TEST_MODEL
)